## Gemini API Key (事先申請)

### 從 `.env` 載入

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
print(f"API Key Found: {os.environ.get('GEMINI_API_KEY') is not None}")

API Key Found: True


### 用原生Gemini確認是否有通

In [2]:
from google import genai
client = genai.Client()
response = client.models.generate_content(model="gemini-2.5-flash", contents="Hello")
print(response.text)

Hello! How can I help you today?


## LangChain + Gemini

### Gemini Setting

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI # 引入 Gemini 聊天模型類別 也可以使用其他LLM 可參考：https://docs.langchain.com/oss/python/integrations/providers/overview
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 選擇模型 (Model)
# 使用 Google 的 Chat 模型，這裡我們選擇快速且高性能的 gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GEMINI_API_KEY"))

# 2. 建立提示模板 (Prompt Template)
prompt_text = "你是一位專業的學術文獻總結專家，請用繁體中文，針對使用者提供的內容，進行精簡且重點突出的摘要。"
# prompt_text = "你是一位專業的學術文獻總結專家，請根據使用者提問時使用的語言，針對提供的內容，進行精簡且重點突出的摘要。"
prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_text),
    ("user", "{text_content}") # {text_content} 是一個變數
])

# 3. 建立輸出解析器 (Output Parser)
output_parser = StrOutputParser()

# 4. 串聯組件 (Chain)
# 整個流程：輸入 -> 提示模板 -> Gemini 模型 -> 輸出解析器
simple_chain = prompt | llm | output_parser

# 5. 執行 Chain
article_text = "Transformer 架構自 2017 年問世以來，已成為自然語言處理和電腦視覺領域的基石。最新的研究傾向於減少注意力機制的計算複雜度，並將其應用擴展到更長的序列任務上。此外，多模態 Transformer 模型的發展，如能處理文本和圖像的模型，正在成為新的研究焦點。"

print("--- 正在呼叫 Gemini 模型進行摘要 ---")
result = simple_chain.invoke({"text_content": article_text})

print("\n--- Gemini 摘要結果 ---")
print(result)

--- 正在呼叫 Gemini 模型進行摘要 ---

--- Gemini 摘要結果 ---
Transformer 架構自 2017 年問世以來，已成為自然語言處理及電腦視覺領域的基石。當前研究趨勢主要聚焦於兩方面：一是降低注意力機制的計算複雜度，以擴展其在長序列任務上的應用；二是發展多模態 Transformer 模型，特別是整合文本與圖像處理能力的模型，作為新的研究焦點。


### Search from ArXiv

In [4]:
import arxiv
from langchain_core.documents import Document
from typing import List

# 💡 請修改您的研究主題和參數 💡
research_topic = "Retrieval-Augmented Generation"  
max_results = 5  

print(f"--- 正在使用純粹 arxiv 套件搜尋主題 '{research_topic}' 的最新 {max_results} 篇論文 ---")

def load_arxiv_documents(query: str, max_results: int) -> List[Document]:
    """使用 arxiv 套件查詢並轉換為 LangChain Document"""
    client = arxiv.Client()
    
    # 建立查詢條件
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate # 依提交日期排序
    )
    
    # 執行查詢
    results = client.results(search)
    
    docs = []
    for result in results:
        # 手動將每個結果轉換為 LangChain Document
        doc = Document(
            # 摘要作為主要內容 (page_content)
            page_content=result.summary,
            # 將其他元數據放入 metadata 字典
            metadata={
                "Title": result.title,
                "Authors": ", ".join([a.name for a in result.authors]),
                "Published": result.published.strftime("%Y-%m-%d"),
                "URL": result.entry_id,
                "PDF_URL": result.pdf_url,
                "Categories": ", ".join(result.categories),
            }
        )
        docs.append(doc)
    return docs

# 執行載入
docs = load_arxiv_documents(research_topic, max_results)

print(f"✅ 成功載入 {len(docs)} 篇論文。")
print(f"首篇論文標題: {docs[0].metadata['Title']}")
print("-" * 30)

# 顯示第一篇文檔的內容和元數據結構
print("--- Document Content (Abstract) ---")
print(docs[0].page_content[:500])
print("\n--- Document Metadata ---")
print(docs[0].metadata)

--- 正在使用純粹 arxiv 套件搜尋主題 'Retrieval-Augmented Generation' 的最新 5 篇論文 ---
✅ 成功載入 5 篇論文。
首篇論文標題: AR-RAG: Autoregressive Retrieval Augmentation for Image Generation
------------------------------
--- Document Content (Abstract) ---
We introduce Autoregressive Retrieval Augmentation (AR-RAG), a novel paradigm that enhances image generation by autoregressively incorporating knearest neighbor retrievals at the patch level. Unlike prior methods that perform a single, static retrieval before generation and condition the entire generation on fixed reference images, AR-RAG performs context-aware retrievals at each generation step, using prior-generated patches as queries to retrieve and incorporate the most relevant patch-level v

--- Document Metadata ---
{'Title': 'AR-RAG: Autoregressive Retrieval Augmentation for Image Generation', 'Authors': 'Jingyuan Qi, Zhiyang Xu, Qifan Wang, Lifu Huang', 'Published': '2025-06-08', 'URL': 'http://arxiv.org/abs/2506.06962v3', 'PDF_URL': None, 'Categories': '

### (X) ArXiv Loader

In [5]:
'''
from langchain_community.document_loaders import ArxivLoader
from langchain_community.utilities import ArxivAPIWrapper

# 💡 請修改您的研究主題和參數 💡
research_topic = "Retrieval-Augmented Generation"  # 例如：RAG 技術
max_results = 5  # 限制只搜尋最新的 5 篇論文

# print(f"--- 正在從 arXiv 搜尋並載入主題 '{research_topic}' 的最新 {max_results} 篇論文 ---")
arxiv_api_wrapper = ArxivAPIWrapper(
    load_max_docs=max_results, 
    # 關鍵設定：設定為 False 避免嘗試下載 PDF
    doc_content_chars_max=0,  
    # 另一個參數 (如果需要)：設定為 False 避免下載整個 PDF (但上面的參數更直接)
    keep_pdf_on_disk=False 
)

loader = ArxivLoader(
    query=research_topic,  
    load_max_docs=max_results,
    # 💡 將自定義的 API wrapper 傳入 loader 中
    arxiv_api_wrapper=arxiv_api_wrapper
)

docs = loader.load()


print(f"✅ 成功載入 {len(docs)} 篇論文。")
print(f"首篇論文標題: {docs[0].metadata['Title']}")
print("-" * 30)

# 顯示第一篇文檔的內容和元數據結構
print(docs[0].page_content[:500]) # 這是摘要/前言部分
print(docs[0].metadata)
'''

'\nfrom langchain_community.document_loaders import ArxivLoader\nfrom langchain_community.utilities import ArxivAPIWrapper\n\n# 💡 請修改您的研究主題和參數 💡\nresearch_topic = "Retrieval-Augmented Generation"  # 例如：RAG 技術\nmax_results = 5  # 限制只搜尋最新的 5 篇論文\n\n# print(f"--- 正在從 arXiv 搜尋並載入主題 \'{research_topic}\' 的最新 {max_results} 篇論文 ---")\narxiv_api_wrapper = ArxivAPIWrapper(\n    load_max_docs=max_results, \n    # 關鍵設定：設定為 False 避免嘗試下載 PDF\n    doc_content_chars_max=0,  \n    # 另一個參數 (如果需要)：設定為 False 避免下載整個 PDF (但上面的參數更直接)\n    keep_pdf_on_disk=False \n)\n\nloader = ArxivLoader(\n    query=research_topic,  \n    load_max_docs=max_results,\n    # 💡 將自定義的 API wrapper 傳入 loader 中\n    arxiv_api_wrapper=arxiv_api_wrapper\n)\n\ndocs = loader.load()\n\n\nprint(f"✅ 成功載入 {len(docs)} 篇論文。")\nprint(f"首篇論文標題: {docs[0].metadata[\'Title\']}")\nprint("-" * 30)\n\n# 顯示第一篇文檔的內容和元數據結構\nprint(docs[0].page_content[:500]) # 這是摘要/前言部分\nprint(docs[0].metadata)\n'

### LangChain Text Splitting

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 設置文本切分器
# chunk_size: 每個文本塊的大小（例如 1000 個字符）
# chunk_overlap: 文本塊之間的重疊量，有助於保持上下文的連續性
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150, 
    add_start_index=True # 添加開始索引，方便追溯原始文件位置
)

# 將文件列表 (docs) 切分成更小的文本塊
all_splits = text_splitter.split_documents(docs)

print(f"原始文檔數量: {len(docs)}")
print(f"切分後的文本塊總數: {len(all_splits)}")
print(f"範例文本塊 (Chunk) 長度: {len(all_splits[0].page_content)} 個字符")

原始文檔數量: 5
切分後的文本塊總數: 11
範例文本塊 (Chunk) 長度: 989 個字符


### Gemini Embedding

In [8]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
# 確保 all_splits 已經在步驟中成功定義

# --- 確保這段程式碼已經運行 ---
# 1. 初始化嵌入模型 (Embedding Model)
# 確保 os.environ.get("GEMINI_API_KEY") 能夠取到金鑰
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print("\n--- 正在創建向量資料庫 ---")

# 2. 創建 FAISS 向量儲存
# 這一步會將 all_splits 向量化，並將結果存入 vectorstore 變數中
vectorstore = FAISS.from_documents(
    documents=all_splits,
    embedding=embedding_model
)

print("✅ 向量資料庫建立成功！知識庫已就緒。")
# -------------------------------


--- 正在創建向量資料庫 ---
✅ 向量資料庫建立成功！知識庫已就緒。


In [9]:
# 根據您先前建立的 FAISS 向量儲存，創建檢索器
# k=3 表示每次查詢時，檢索器會返回最相關的 3 個文本塊
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ 檢索器創建成功。")

✅ 檢索器創建成功。


### RAG

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 定義 RAG 提示模板
template = """你是一位專業的研究助理，請根據以下提供的『上下文資料』，詳細且準確地回答用戶的問題。
請僅使用上下文中的資訊來回答，如果資訊不足，請表明「資料不足，無法回答」。
請使用繁體中文。

--- 上下文資料 ---
{context}

--- 使用者問題 ---
{question}
"""

RAG_PROMPT = ChatPromptTemplate.from_template(template)

print("✅ RAG 提示模板定義完成。")

✅ RAG 提示模板定義完成。


In [11]:
# 確保 Gemini 模型已初始化
# 假設 llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=...) 已經在前面的步驟中成功運行

# 定義一個格式化函數，用於將檢索到的文檔（Documents）轉換成單一的字串（Context）
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 串聯完整的 RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ 完整的 RAG 檢索與生成鏈（RAG Chain）建構完成。")

✅ 完整的 RAG 檢索與生成鏈（RAG Chain）建構完成。


### 提問

In [12]:
# 💡 請替換為您的研究問題 💡
question = "Retrieval-Augmented Generation (RAG) 技術有哪些主要的優化方向？根據最新的論文，哪種優化方法的效果最好？"

print(f"\n--- 正在向知識庫查詢問題：{question} ---")

# 運行 RAG Chain
# Chain 會自動：檢索 -> 構建提示 -> 傳給 Gemini -> 獲取最終答案
response = rag_chain.invoke(question)

print("\n--- 深度研究報告摘要 ---")
print(response)


--- 正在向知識庫查詢問題：Retrieval-Augmented Generation (RAG) 技術有哪些主要的優化方向？根據最新的論文，哪種優化方法的效果最好？ ---

--- 深度研究報告摘要 ---
根據提供的上下文資料，Retrieval-Augmented Generation (RAG) 技術有以下主要的優化方向：

1.  **AR-RAG (針對圖像生成模型)**：
    *   整合模型預測補丁（model-predicted patches）的分布與檢索到的補丁（retrieved patches）的分布。
    *   特徵增強解碼（Feature-Augmentation in Decoding, FAiD）：一種參數高效的微調方法，透過多尺度卷積操作逐步平滑檢索到的補丁特徵，並利用這些特徵來增強圖像生成過程。

2.  **FAIR-RAG (針對複雜、多跳查詢)**：
    *   引入一個代理框架（agentic framework），將標準RAG管線轉變為動態的、證據驅動的推理過程。
    *   核心是透過結構化證據評估（Structured Evidence Assessment, SEA）模組來管理迭代細化循環（Iterative Refinement Cycle）。SEA能將初始查詢分解為所需發現的清單，並審核聚合的證據以識別已確認的事實和資訊空白，這些空白提供了精確的信號來填補。

3.  **EVOR (可組合的靈活方法)**：
    *   靈活且可以與其他方法（如Self-RAG、In-context RAG、DocPrompting）結合使用，以實現進一步的改進。
    *   受益於查詢和文件的同步演變，以及知識庫中多樣化的資訊來源。

根據最新的論文，在所提供的資料中，**AR-RAG** 證明了其在廣泛採用的基準測試（包括Midjourney-30K、GenEval和DPG-Bench）上，相較於最先進的圖像生成模型，展現了**顯著的性能提升（significant performance gains）**。
